# Paper 3 raw family labels and temporal coverage

Open in Google Colab with a **CPU runtime**, then choose **Runtime → Run all**. No GitHub clone or model loading is required.

This next step reads all four original zipped CSVs. It recovers attack-family labels and timestamps for the saved source/candidate row IDs, verifies their binary labels against saved inputs, and counts full-capture classes, exact family names and 3/6/12/24-hour UTC blocks. It does **not** train models or produce new detector/warning performance results.

Outputs go to **Paper3_DataSet → supplementary_raw_metadata_recovery_20261024 → run_v1**. Existing experiment files are read only. Each dataset has its own completed checkpoint. If interrupted, rerun the same notebook: completed datasets are verified and reused; an unfinished dataset restarts from its beginning. Do not alter the code or configuration mid-run.

The saved per-row metadata lookup stays in your Google Drive for later analyses. The downloaded report ZIP contains aggregate counts, checksums and code; it excludes per-row lookups, IP addresses, raw predictor rows and models.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
PAPER3_ROOT = Path('/content/drive/MyDrive/research/researchdata/NIDS_Research/Paper3_DataSet')
RUN_DIR = PAPER3_ROOT / 'supplementary_raw_metadata_recovery_20261024' / 'run_v1'
if not PAPER3_ROOT.is_dir():
    raise FileNotFoundError(f'Check PAPER3_ROOT: {PAPER3_ROOT}')
print('Output folder:', RUN_DIR)

## Full raw metadata scan

This is more work than the readiness audit: all raw CSV rows must be decompressed and parsed. The first run may take tens of minutes or longer, depending on Colab CPU and Drive speed; there is no measured runtime estimate yet. Progress appears every 2.5 million rows. Keep the runtime connected.

The scan records a post hoc plan before calculations. It checks the raw member name, archive size and complete parsed row count against the original candidate manifest; recorded hashes of all saved labels/source partitions must match. New raw archive hashes are reference hashes, not historical seals. Row IDs use the original zero-based parsed CSV row order. All selected binary labels are compared, but all 49 feature values are not replayed.

Observed class proportions describe the benchmark captures, not operational deployment. Raw temporal counts precede source/duplicate exclusions; they do not alone establish usable experimental panels. Family names stay separate by dataset until their definitions can be assessed.

In [ ]:
RECOVERY_SOURCE = '"""Raw metadata recovery and descriptive inventory; no fitted warning or detector evaluation."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nfrom collections import Counter\nimport hashlib, json, zipfile, platform\nimport numpy as np\nimport pandas as pd\n\nDATASETS=[\'NF_BoT_IoT_v3\',\'NF_CSE_CIC_IDS2018_v3\',\'NF_ToN_IoT_v3\',\'NF_UNSW_NB15_v3\']\nWIDTHS=[3,6,12,24]\n\ndef sha(p):\n    h=hashlib.sha256()\n    with Path(p).open(\'rb\') as f:\n        for b in iter(lambda:f.read(2**20),b\'\'):h.update(b)\n    return h.hexdigest()\n\ndef write_json(p,value):\n    p=Path(p);q=p.with_suffix(\'.tmp\')\n    q.write_text(json.dumps(value,indent=2,allow_nan=False)+\'\\n\');q.replace(p)\n\ndef read_json(p):return json.loads(Path(p).read_text())\n\ndef label_input(p,expected,role):\n    actual=sha(p)\n    if not expected or actual!=expected:raise ValueError(f\'Missing/mismatched recorded hash: {p}\')\n    f=pd.read_parquet(p,columns=[\'row_id\',\'Label\'])\n    ids=pd.to_numeric(f.row_id,errors=\'coerce\').to_numpy(float)\n    if not (np.isfinite(ids).all() and (ids>=0).all() and (ids==np.floor(ids)).all() and f.row_id.is_unique):\n        raise ValueError(f\'Invalid row IDs: {p}\')\n    if not f.Label.isin([0,1]).all():raise ValueError(f\'Invalid labels: {p}\')\n    return f,{\'role\':role,\'rows\':len(f),\'sha256\':actual}\n\ndef recover_all(root,run_dir,chunk_size=250000):\n    root=Path(root).resolve();run_dir=Path(run_dir).resolve()\n    if not root.is_dir():raise FileNotFoundError(root)\n    if run_dir.parent!=root/\'supplementary_raw_metadata_recovery_20261024\':\n        raise ValueError(\'Use the separate supplementary_raw_metadata_recovery_20261024 run folder\')\n    run_dir.mkdir(parents=True,exist_ok=True)\n    plan=\'\'\'Post hoc input recovery plan recorded before this raw scan.\nRecover Attack and FLOW_START_MILLISECONDS only for the union of saved source and candidate row IDs.\nCheck every recovered binary label against every saved input label; check complete CSV row count and archive member/size against the candidate manifest.\nCount full raw binary labels, exact family strings and UTC-aligned 3/6/12/24-hour blocks. Eligibility count uses the historical quota of at least 2,000 rows per class per block and is descriptive only.\nRetain family names separately by dataset; no semantic harmonisation, model fitting, target accuracy, warning association or significance testing.\nRaw observed class proportions describe these benchmark captures, not production prevalence. Counts do not construct leakage-free evaluation panels.\nRow IDs follow zero-based parsed-data-row order used by the original pandas reader. Binary-label agreement and input hashes are checked, but all 49 raw feature values are not replayed.\nAll original model, panel, prediction and seal files are read only. New local lookup files stay in Drive; the download ZIP contains aggregates, hashes and code only.\nCompleted datasets may be reused only after current input hashes and saved output hashes agree. An interrupted dataset restarts from its beginning in a new attempt folder.\n\'\'\'\n    config={\'datasets\':DATASETS,\'width_hours\':WIDTHS,\'chunk_size\':chunk_size,\'rows_per_class_quota\':2000,\n            \'pandas_version\':pd.__version__,\'numpy_version\':np.__version__,\n            \'code_sha256\':hashlib.sha256(RECOVERY_SOURCE.encode()).hexdigest(),\'plan_sha256\':hashlib.sha256(plan.encode()).hexdigest()}\n    identity=run_dir/\'run_identity.json\'\n    if identity.exists():\n        if read_json(identity)[\'configuration\']!=config:raise ValueError(\'Run configuration/code differs; use a new run folder\')\n    else:\n        write_json(identity,{\'created_utc\':datetime.now(timezone.utc).isoformat(),\'configuration\':config})\n        (run_dir/\'PLAN.md\').write_text(plan);(run_dir/\'RECOVERY_CODE.py\').write_text(RECOVERY_SOURCE)\n    results=[];errors=[]\n    for dataset in DATASETS:\n        print(\'\\nChecking raw recovery inputs:\',dataset,flush=True)\n        folder=run_dir/dataset;folder.mkdir(exist_ok=True)\n        try:\n            mpath=root/\'Paper3_Target_Panels/candidates\'/dataset/\'candidate_manifest.json\'\n            m=read_json(mpath)\n            archive=root/m[\'raw_zip_name\']\n            if archive.parent!=root or not archive.is_file():raise ValueError(\'Raw archive path is not an existing file in Paper 3 root\')\n            if archive.stat().st_size!=m[\'raw_zip_size_bytes\']:raise ValueError(\'Raw archive size differs from original candidate manifest\')\n            inputs={\'candidate_manifest\':sha(mpath),\'raw_archive\':sha(archive)}\n            roles=[]\n            p=root/\'Paper3_Target_Label_Vault\'/dataset/\'target_candidate_labels.parquet\'\n            f,rec=label_input(p,m.get(\'label_vault_sha256\'),\'target_candidate\')\n            roles.append((rec,f));inputs[str(p.relative_to(root))]=rec[\'sha256\']\n            for seed in range(2027,2032):\n                base=root/\'Paper3_Sanitized\'/dataset/f\'seed_{seed}\'\n                sm=read_json(base/\'sanitization_manifest.json\')\n                inputs[str((base/\'sanitization_manifest.json\').relative_to(root))]=sha(base/\'sanitization_manifest.json\')\n                for split in [\'train\',\'calibration\',\'source_test\',\'environment\']:\n                    p=base/(split+\'.parquet\')\n                    f,rec=label_input(p,sm.get(\'output_files\',{}).get(split,{}).get(\'sha256\'),f\'seed_{seed}/{split}\')\n                    roles.append((rec,f));inputs[str(p.relative_to(root))]=rec[\'sha256\']\n            joined=pd.concat([f for _,f in roles],ignore_index=True).sort_values(\'row_id\',kind=\'stable\')\n            if (joined.groupby(\'row_id\').Label.nunique()>1).any():raise ValueError(\'Saved labels conflict across roles/seeds\')\n            joined=joined.drop_duplicates(\'row_id\').reset_index(drop=True)\n            wanted=joined.row_id.to_numpy(np.int64)\n            marker=folder/\'complete.json\'\n            if marker.exists():\n                cached=read_json(marker)\n                if cached[\'input_hashes\']!=inputs:raise ValueError(\'Current inputs differ from completed dataset checkpoint\')\n                for rel,digest in cached[\'output_hashes\'].items():\n                    if sha(folder/rel)!=digest:raise ValueError(\'Completed recovery output failed hash verification: \'+rel)\n                results.append(cached);print(\'REUSED verified completed dataset:\',dataset,flush=True);continue\n            attempt=folder/datetime.now(timezone.utc).strftime(\'attempt_%Y%m%d_%H%M%S_%f\');attempt.mkdir()\n            write_json(attempt/\'input_hashes.json\',inputs)\n            write_json(attempt/\'progress.json\',{\'status\':\'RUNNING\',\'rows_scanned\':0})\n            pieces=[];counts=Counter();family_counts=Counter();temporal=Counter()\n            offset=0;invalid_labels=0;invalid_timestamps=0;missing_families=0\n            timestamp_min=None;timestamp_max=None\n            with zipfile.ZipFile(archive) as z:\n                member=m[\'raw_csv_member\'];info=z.getinfo(member)\n                with z.open(info) as stream:\n                    columns=pd.read_csv(stream,nrows=0).columns.tolist()\n                required=[\'Label\',\'Attack\',\'FLOW_START_MILLISECONDS\']\n                if not set(required).issubset(columns):raise ValueError(\'Raw CSV lacks required metadata columns\')\n                with z.open(info) as stream:\n                    reader=pd.read_csv(stream,usecols=required,chunksize=chunk_size,low_memory=False,\n                        dtype={\'Attack\':\'string\'},on_bad_lines=\'error\')\n                    for chunk_no,chunk in enumerate(reader,1):\n                        n=len(chunk);labels=pd.to_numeric(chunk.Label,errors=\'coerce\').to_numpy(float)\n                        good_label=np.isin(labels,[0,1]);invalid_labels+=int((~good_label).sum())\n                        label_names=np.where(labels==0,\'0\',np.where(labels==1,\'1\',\'INVALID\'))\n                        fam=chunk.Attack\n                        missing_families+=int((fam.isna()|fam.fillna(\'\').str.strip().eq(\'\')).sum())\n                        # Missing is a separate flag/category, not a replacement family name.\n                        family_frame=pd.DataFrame({\'label\':label_names,\'family\':fam.fillna(\'\'),\'family_missing\':fam.isna()})\n                        for (label,family,missing),v in family_frame.groupby([\'label\',\'family\',\'family_missing\'],dropna=False).size().items():\n                            family_counts[(str(label),str(family),bool(missing))]+=int(v)\n                        counts.update({str(k):int(v) for k,v in pd.Series(label_names).value_counts().items()})\n                        timestamps=pd.to_numeric(chunk.FLOW_START_MILLISECONDS,errors=\'coerce\').to_numpy(float)\n                        good_ts=np.isfinite(timestamps)&(timestamps>=0)&(timestamps<9e18)&(timestamps==np.floor(timestamps))\n                        invalid_timestamps+=int((~good_ts).sum())\n                        if good_ts.any():\n                            lo=int(timestamps[good_ts].min());hi=int(timestamps[good_ts].max())\n                            timestamp_min=lo if timestamp_min is None else min(timestamp_min,lo)\n                            timestamp_max=hi if timestamp_max is None else max(timestamp_max,hi)\n                        mask=good_ts&good_label\n                        valid_ts=timestamps[mask].astype(np.int64);valid_lab=labels[mask].astype(int)\n                        for hours in WIDTHS:\n                            width=hours*3600000\n                            tmp=pd.DataFrame({\'start\':valid_ts//width*width,\'label\':valid_lab})\n                            for (start,label),v in tmp.groupby([\'start\',\'label\']).size().items():\n                                temporal[(hours,int(start),int(label))]+=int(v)\n                        left=np.searchsorted(wanted,offset);right=np.searchsorted(wanted,offset+n)\n                        if right>left:\n                            chosen=wanted[left:right];loc=chosen-offset\n                            recovered=chunk.iloc[loc][required].copy()\n                            recovered.insert(0,\'row_id\',chosen)\n                            recovered[\'Label\']=labels[loc]\n                            recovered[\'FLOW_START_MILLISECONDS\']=timestamps[loc]\n                            recovered[\'timestamp_valid\']=good_ts[loc]\n                            pieces.append(recovered)\n                        offset+=n\n                        write_json(attempt/\'progress.json\',{\'status\':\'RUNNING\',\'rows_scanned\':offset,\'chunks_scanned\':chunk_no,\n                            \'selected_rows_recovered\':sum(len(x) for x in pieces)})\n                        if chunk_no%10==0:print(dataset,format(offset,\',\'),\'raw rows scanned\',flush=True)\n            if offset!=int(m[\'rows_scanned\']):raise ValueError(f\'Raw row count {offset} differs from original {m["rows_scanned"]}\')\n            if invalid_labels!=int(m[\'invalid_labels\']):raise ValueError(\'Invalid raw label count differs from original candidate scan\')\n            recovered=pd.concat(pieces,ignore_index=True).sort_values(\'row_id\') if pieces else pd.DataFrame()\n            if len(recovered)!=len(wanted) or not np.array_equal(recovered.row_id.to_numpy(),wanted):\n                raise ValueError(\'Not every saved row ID was recovered exactly once\')\n            if not np.array_equal(recovered.Label.to_numpy(float),joined.Label.to_numpy(float)):\n                raise ValueError(\'Recovered raw binary labels disagree with saved input labels\')\n            recovered[\'Label\']=recovered.Label.astype(\'int8\')\n            recovered.to_parquet(attempt/\'saved_row_metadata.parquet\',index=False)\n            lookup=recovered.set_index(\'row_id\',verify_integrity=True)\n            role_counts=[]\n            for rec,f in roles:\n                sub=lookup.loc[f.row_id]\n                if not np.array_equal(sub.Label.to_numpy(),f.Label.to_numpy()):raise ValueError(\'Role-specific label alignment failed\')\n                temp=pd.DataFrame({\'Label\':sub.Label.to_numpy(),\'Attack\':sub.Attack.fillna(\'\').to_numpy(),\n                    \'family_missing\':sub.Attack.isna().to_numpy()})\n                for (label,family,missing),v in temp.groupby([\'Label\',\'Attack\',\'family_missing\'],dropna=False).size().items():\n                    role_counts.append({\'dataset\':dataset,\'role\':rec[\'role\'],\'label\':int(label),\'family\':str(family),\'family_missing\':bool(missing),\'rows\':int(v)})\n            pd.DataFrame(role_counts).to_csv(attempt/\'saved_role_family_counts.csv\',index=False)\n            pd.DataFrame([{\'dataset\':dataset,\'label\':l,\'family\':f,\'family_missing\':m,\'rows\':v} for (l,f,m),v in sorted(family_counts.items())]).to_csv(attempt/\'raw_family_counts.csv\',index=False)\n            tr=pd.DataFrame([{\'dataset\':dataset,\'hours\':h,\'group_start_ms\':t,\'label\':l,\'rows\':v} for (h,t,l),v in sorted(temporal.items())])\n            tr.to_csv(attempt/\'raw_temporal_counts.csv\',index=False)\n            coverage=[]\n            for h in WIDTHS:\n                q=tr[tr.hours==h].pivot(index=\'group_start_ms\',columns=\'label\',values=\'rows\').fillna(0).reindex(columns=[0,1],fill_value=0)\n                coverage.append({\'hours\':h,\'observed_blocks_with_valid_binary_labels\':len(q),\'blocks_with_2000_each_class\':int(((q[0]>=2000)&(q[1]>=2000)).sum()),\n                    \'counts_before_source_or_duplicate_exclusion\':True})\n            valid_total=counts[\'0\']+counts[\'1\']\n            summary={\'dataset\':dataset,\'status\':\'RAW_METADATA_RECOVERED\',\'raw_rows\':offset,\'raw_class_counts\':dict(counts),\n                \'observed_capture_attack_fraction_among_valid_binary_labels\':counts[\'1\']/valid_total if valid_total else None,\n                \'invalid_raw_labels\':invalid_labels,\'invalid_raw_timestamps\':invalid_timestamps,\n                \'minimum_valid_timestamp_ms\':timestamp_min,\'maximum_valid_timestamp_ms\':timestamp_max,\n                \'missing_or_blank_raw_families\':missing_families,\'saved_unique_rows_recovered\':len(recovered),\n                \'saved_roles_checked\':len(roles),\'saved_raw_binary_labels_all_match\':True,\n                \'saved_rows_invalid_timestamps\':int((~recovered.timestamp_valid).sum()),\'temporal_coverage\':coverage,\n                \'all_49_raw_features_replayed\':False,\'new_detector_or_warning_experiments_completed\':0,\n                \'raw_zip_sha256_is_new_reference_not_original_historical_seal\':True,\n                \'meaning\':\'Full capture counts are benchmark descriptions, not production prevalence. Family strings remain dataset-specific. Temporal counts precede leakage exclusions.\'}\n            write_json(attempt/\'summary.json\',summary)\n            write_json(attempt/\'progress.json\',{\'status\':\'COMPLETED\',\'rows_scanned\':offset})\n            hashes={str(p.relative_to(folder)):sha(p) for p in sorted(attempt.iterdir()) if p.is_file()}\n            cached={\'dataset\':dataset,\'input_hashes\':inputs,\'output_hashes\':hashes,\'attempt\':attempt.name,\'summary\':summary}\n            write_json(marker,cached);results.append(cached)\n            print(\'Recovered and checked:\',dataset,\'| unique saved rows:\',len(recovered),flush=True)\n        except Exception as e:\n            errors.append({\'dataset\':dataset,\'error\':str(e)})\n            print(\'DATASET STOPPED:\',dataset,str(e),flush=True)\n    final={\'status\':\'RECOVERY_COMPLETED\' if len(results)==len(DATASETS) and not errors else \'INCOMPLETE_RECOVERY\',\n        \'created_utc\':datetime.now(timezone.utc).isoformat(),\'datasets_completed\':len(results),\'datasets_expected\':len(DATASETS),\n        \'summaries\':[x[\'summary\'] for x in results],\'errors\':errors,\'python\':platform.python_version(),\n        \'new_performance_results\':False,\'lookup_files_remain_in_drive\':True}\n    write_json(run_dir/\'recovery_summary.json\',final)\n    # Small download bundle: no per-row lookup files, raw flows or model objects.\n    package=run_dir/(\'recovery_report_\'+datetime.now(timezone.utc).strftime(\'%Y%m%d_%H%M%S_%f\')+\'.zip\')\n    with zipfile.ZipFile(package,\'x\',zipfile.ZIP_DEFLATED) as z:\n        for name in [\'PLAN.md\',\'RECOVERY_CODE.py\',\'run_identity.json\',\'recovery_summary.json\']:\n            z.write(run_dir/name,name)\n        for cached in results:\n            folder=run_dir/cached[\'dataset\'];z.write(folder/\'complete.json\',cached[\'dataset\']+\'/complete.json\')\n            for name in [\'summary.json\',\'raw_family_counts.csv\',\'raw_temporal_counts.csv\',\'saved_role_family_counts.csv\',\'progress.json\']:\n                p=folder/cached[\'attempt\']/name;z.write(p,cached[\'dataset\']+\'/\'+name)\n    print(\'\\nStatus:\',final[\'status\']);print(\'Report ZIP:\',package)\n    if errors:print(\'Read the errors in recovery_summary.json; completed datasets will be verified and reused on retry.\')\n    return package\n'
exec(compile(RECOVERY_SOURCE, "paper3_raw_metadata_recovery.py", "exec"))

In [ ]:
report_zip = recover_all(PAPER3_ROOT, RUN_DIR, chunk_size=250000)
from google.colab import files
files.download(str(report_zip))

## Send the output back

Upload the downloaded **recovery_report_….zip** to our chat. `RECOVERY_COMPLETED` means all four metadata scans completed; it does not mean the pending experiments are finished.

If the status is `INCOMPLETE_RECOVERY`, send that report ZIP too. It contains the errors. Keep the Drive lookup files and checkpoints unchanged. If the browser download does not start, download the newest `recovery_report_….zip` from the printed `run_v1` folder in Drive.

After reviewing the full family definitions/support and timestamps, we can prepare the family-specific evaluation and separately plan fresh temporal and natural-prevalence panels. No such model evaluation occurs in this notebook.